# Phase 5: Master dataset

In this notebook we combine the results of Phase 3 and Phase 4 into one table. Each row is one neighbourhood, with its car ownership, its nearest station, the station type, the number of disruptions at that station, and the control variables. This table is used for all maps and models.

In [1]:
import os                  # for file paths
import pandas as pd        # for tables

# Folder with the clean data from the earlier phases
clean_folder = 'data/cleaned_data'

# Load the neighbourhoods with their nearest station (Phase 3) and the disruptions per station (Phase 4)
buurten_met_station = pd.read_pickle(os.path.join(clean_folder, 'buurten_met_station.pkl'))
storingen_per_station = pd.read_pickle(os.path.join(clean_folder, 'storingen_per_station.pkl'))

print('buurten_met_station:', len(buurten_met_station), 'rows')
print('storingen_per_station:', len(storingen_per_station), 'rows')

buurten_met_station: 11221 rows
storingen_per_station: 397 rows


## Step 1: Add the disruptions to every neighbourhood

In [2]:
# Every neighbourhood has the code of its nearest station.
# We look up that station code in the disruption table and add its disruption numbers to the neighbourhood.
# Neighbourhoods with the same nearest station get the same disruption numbers.
master = buurten_met_station.merge(storingen_per_station, on='station_code')

# Check: we should still have the same number of neighbourhoods as before
print('Neighbourhoods:', len(master))

# Check: count the empty cells in every column (isna() is True for an empty cell, sum() counts the Trues)
master.isna().sum()

Neighbourhoods: 11202


buurtcode                  0
buurtnaam                  0
gemeente                   0
autos_per_hh               0
inkomen                    0
stedelijkheid              0
hh_grootte                 0
dichtheid                  0
n_huishoudens              0
n_bedrijven                0
afstand_station            0
afstand_overstapstation    0
bedrijven_per_hh           0
station_code               0
station_naam               0
station_type               0
station_groot              0
storingen_per_jaar         0
minuten_per_jaar           0
dtype: int64

## Step 2: Keep only the columns we need

In [3]:
# The columns of the master table, in a logical order
kolommen = [
    'buurtcode', 'buurtnaam', 'gemeente',                              # which neighbourhood
    'autos_per_hh',                                                     # car ownership in that neighbourhood
    'afstand_station', 'afstand_overstapstation',                       # distance to the nearest (transfer) station in km
    'station_code', 'station_naam', 'station_type', 'station_groot',   # which station is nearest, and is it a major station
    'storingen_per_jaar', 'minuten_per_jaar',                           # how often that station has disruptions
    'inkomen', 'stedelijkheid', 'hh_grootte', 'dichtheid',              # control variables
]
master = master[kolommen]

master.head()

,buurtcode,buurtnaam,gemeente,autos_per_hh,afstand_station,afstand_overstapstation,station_code,station_naam,station_type,station_groot,storingen_per_jaar,minuten_per_jaar,inkomen,stedelijkheid,hh_grootte,dichtheid
0,BU16800000,Annen,Aa en Hunze,1.3,13.2,14.2,MTH,Martenshoek,stoptreinstation,False,12.333333,3416.333333,33.4,5.0,2.3,1369.0
1,BU16800009,Verspreide huizen Annen,Aa en Hunze,1.5,13.7,15.3,VDM,Veendam,knooppuntStoptreinstation,False,6.333333,1887.333333,49.5,5.0,2.3,12.0
2,BU16800100,Eext,Aa en Hunze,1.4,13.6,13.6,ASN,Assen,knooppuntIntercitystation,True,38.000000,4385.333333,33.2,5.0,2.3,746.0
3,BU16800109,Verspreide huizen Eext,Aa en Hunze,1.1,15.4,15.6,ASN,Assen,knooppuntIntercitystation,True,38.000000,4385.333333,35.5,5.0,2.0,11.0
4,BU16800200,Anloo,Aa en Hunze,1.3,11.8,11.8,ASN,Assen,knooppuntIntercitystation,True,38.000000,4385.333333,34.2,5.0,2.3,569.0


## Step 3: A first look at the numbers

In [ ]:
# Summary statistics of every number column: count, mean, standard deviation, minimum, maximum, etc. this was already done in Phase 1, but we do it again to check that the numbers are still the same after merging with the station data.
master.describe()

,autos_per_hh,afstand_station,afstand_overstapstation,storingen_per_jaar,minuten_per_jaar,inkomen,stedelijkheid,hh_grootte,dichtheid
count,11202.000000,11202.000000,11202.000000,11202.000000,11202.000000,11202.000000,11202.000000,11202.000000,11202.000000
mean,1.162123,6.389600,13.007115,66.891865,13726.394900,33.611105,3.344046,2.248036,3895.393055
std,0.358440,6.820463,9.921809,62.594136,14041.782429,7.401417,1.504926,0.377302,4075.844872
min,0.000000,0.100000,0.100000,2.000000,355.666667,7.400000,1.000000,1.000000,4.000000
25%,0.900000,2.000000,5.300000,27.333333,4686.333333,29.200000,2.000000,2.000000,665.500000
50%,1.200000,4.200000,10.700000,45.333333,10390.666667,32.600000,4.000000,2.300000,3048.000000
75%,1.400000,8.300000,18.300000,90.000000,15891.333333,36.500000,5.000000,2.500000,5723.750000
max,2.500000,58.000000,72.500000,535.333333,100046.333333,108.000000,5.000000,3.800000,55375.000000


## Saving

In [5]:
# Save the master table: all maps and models (Phase 6 and 7) start from this file
master.to_pickle(os.path.join(clean_folder, 'master.pkl'))